# What is Hashing?

Imagine a library with infinite books but only **10 shelves**.
We need a mathematical formula that takes a book ID (a big number) and tells us exactly which shelf (0-9) to put it on.

This formula is called a **Hash Function**.

### The Goal
- **Input:** Any integer (e.g., `104`, `599`, `42`).
- **Output:** A fixed index (e.g., `4`, `9`, `2`).
- **Speed:** $O(1)$ (Instant access).

In [101]:
def simplest_hash(number, tables_size):
    return number % table_size

In [102]:
def simple_hash(number, table_size):
    k_prim = 1
    table_size = table_size ** table_size 
    p = 601
    return (((number * k_prim) % 601) % table_size)

In [103]:
data = [15, 25, 35, 42]
table_size = len(data) 

In [104]:
for x in data:
    shelf = simplest_hash(x, table_size)
    print(f"Number {x} -> goes to shelf {shelf}")

Number 15 -> goes to shelf 3
Number 25 -> goes to shelf 1
Number 35 -> goes to shelf 3
Number 42 -> goes to shelf 2


In [105]:
for x in data:
    shelf = simple_hash(x, table_size)
    print(f"Number {x} -> goes to shelf {shelf}")

Number 15 -> goes to shelf 15
Number 25 -> goes to shelf 25
Number 35 -> goes to shelf 35
Number 42 -> goes to shelf 42


# The Problem: Collisions
With `x%10`, all the books with id-s `[15, 25, 35]` will ended up on **Shelf 5**.
This is called a **Collision**. 

If we just pile them up in a list, finding a specific number becomes slow ($O(N)$). 
To get true $O(1)$ speed, we need a smarter way to organize these "crowded" shelves.

# The Solution: FKS Perfect Hashing
We will use a **Two-Level** approach:
1. **Level 1:** Distribute items into buckets (Some collisions are okay).
2. **Level 2:** For every crowded bucket, build a **secondary table** that is guaranteed to be collision-free.

In [106]:
import random
import unittest
from typing import List, Optional, Tuple

In [107]:
def is_prime(num) -> bool:
    if num < 2: return False
    for i in range(2, int(num ** 0.5) +1):
        if num % i == 0:            
            return False
    
    return True

In [108]:
def next_prime(n:int) -> int:
    candidate = n + 1
    while not is_prime(candidate):
        candidate += 1
    return candidate

In [109]:
PRINT_INFO = True

$$hash(x) = ((k \cdot x) \mod p) \mod n$$

In [124]:
class FKSPerfectHash:
    def __init__(self, numbers: List[int]):
        self.elements = list(set(numbers))
        self.n = len(self.elements)

        if self.n == 0:
            self.p = 1
            self.buckets = []
            self.level2_tables = []
            return

        max_val = max(self.elements)
        self.p = next_prime(max_val)

        if PRINT_INFO: print(f'max_val -> {max_val}, self.p -> {self.p}')

        self._build_level_1()
        self._build_level_2()

    def _build_level_1(self):
        while True:
            self.k_main = random.randint(1, self.p - 1)
            self.buckets = [[] for _ in range(self.n)]

            for x in self.elements:
                idx = ((self.k_main * x) % self.p ) % self.n
                self.buckets[idx].append(x)
    
            sum_squares = sum(len(b)**2 for b in self.buckets)

            if sum_squares < 3 * self.n:
                if PRINT_INFO: print(f'Step 1 ({sum_squares})(n-> {self.n})(k_main -> {self.k_main}) => {self.buckets}')
                break
    
    def _build_level_2(self):
        self.level2_tables = [None] * self.n

        for i, bucket in enumerate(self.buckets):
            b_size = len(bucket)

            if b_size == 0:
                self.level2_tables[i] = (0, 0, None)
                continue
            
            m_j = b_size * b_size
            while True:
                k_prim = random.randint(1, self.p-1)
                temp_storage = [None] * m_j
                collision_found = False

                for x in bucket:
                    idx = ((k_prim * x) % self.p) % m_j

                    if temp_storage[idx] is not None:
                        collision_found = True
                        break
                    temp_storage[idx] = x
                
                if not collision_found:
                    # Success! WE found a perfect configuration for bucket level 2!
                    self.level2_tables[i] = (k_prim, m_j, temp_storage)
                    if PRINT_INFO: print(f'Step 2 ({self.level2_tables})')
                    break
                    
    def query(self, x:int) -> bool:
        if self.n == 0:
            return False

        # Level 1 - Find the bucket!
        idx_1 = ((self.k_main * x) % self.p) %self.n

        # Level 2 -> Retrieve Level 2 parameters for that bucket!
        if PRINT_INFO:
            print("-"* 12)
            print(idx_1)
            print(f'from query self.level2_tables -> {self.level2_tables}')
        
        params = self.level2_tables[idx_1]
        k_prim, m_j, storage = params

        if PRINT_INFO:
            print(f'k_prim -> {k_prim} || slots_in_that_bucket -> {m_j} || storage -> {storage}')

        if storage is None:
            return False
            #raise Exception(f"Error on value {x}!")

        idx_2 = ((k_prim * x) % self.p ) % m_j
        return storage[idx_2] == x

In [125]:
print("-"* 12)

------------


In [126]:
data = [2, 4, 5, 15, 18, 30]
fks = FKSPerfectHash(data)
fks.query(5)

max_val -> 30, self.p -> 31
Step 1 (8)(n-> 6)(k_main -> 16) => [[5], [2], [4], [18, 30], [], [15]]
Step 2 ([(8, 1, [5]), None, None, None, None, None])
Step 2 ([(8, 1, [5]), (5, 1, [2]), None, None, None, None])
Step 2 ([(8, 1, [5]), (5, 1, [2]), (12, 1, [4]), None, None, None])
Step 2 ([(8, 1, [5]), (5, 1, [2]), (12, 1, [4]), (6, 4, [None, 30, None, 18]), None, None])
Step 2 ([(8, 1, [5]), (5, 1, [2]), (12, 1, [4]), (6, 4, [None, 30, None, 18]), (0, 0, None), (17, 1, [15])])
------------
0
from query self.level2_tables -> [(8, 1, [5]), (5, 1, [2]), (12, 1, [4]), (6, 4, [None, 30, None, 18]), (0, 0, None), (17, 1, [15])]
k_prim -> 8 || slots_in_that_bucket -> 1 || storage -> [5]


True

In [127]:
class TestFKSAlgorithm(unittest.TestCase):
    
    def test_paper_example(self):
        if PRINT_INFO: print("\n" + "test_paper_example:")
        data = [2, 4, 5, 15, 18, 30]
        fks = FKSPerfectHash(data)
        
        for x in data:
            self.assertTrue(fks.query(x), f"Should find {x}")
        
        self.assertFalse(fks.query(3))
        self.assertFalse(fks.query(31))
        self.assertFalse(fks.query(100))

    def test_collisions_handling(self):
        if PRINT_INFO: print("\n" + "test_collisions_handling:")
        data = [10, 20, 30, 40, 50, 60, 70, 90, 100] 
        fks = FKSPerfectHash(data)
        
        for x in data:
            self.assertTrue(fks.query(x))

    def test_large_random_dataset(self):
        n = 30
        data = [random.randint(1, 1_000_000) for _ in range(n)]
        unique_data = list(set(data))
        fks = FKSPerfectHash(unique_data)
        for x in unique_data:
            self.assertTrue(fks.query(x))
        self.assertFalse(fks.query(-1))

    def test_single_element(self):
        fks = FKSPerfectHash([42])
        self.assertTrue(fks.query(42))
        self.assertFalse(fks.query(99))

    def test_empty_set(self):
        fks = FKSPerfectHash([])
        self.assertFalse(fks.query(1))

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.007s

OK



test_collisions_handling:
max_val -> 100, self.p -> 101
Step 1 (17)(n-> 9)(k_main -> 52) => [[60, 30], [], [], [50, 20], [100, 70], [], [40, 10], [90], []]
Step 2 ([(91, 4, [None, None, 60, 30]), None, None, None, None, None, None, None, None])
Step 2 ([(91, 4, [None, None, 60, 30]), (0, 0, None), (0, 0, None), (89, 4, [None, None, 50, 20]), None, None, None, None, None])
Step 2 ([(91, 4, [None, None, 60, 30]), (0, 0, None), (0, 0, None), (89, 4, [None, None, 50, 20]), (76, 4, [70, 100, None, None]), None, None, None, None])
Step 2 ([(91, 4, [None, None, 60, 30]), (0, 0, None), (0, 0, None), (89, 4, [None, None, 50, 20]), (76, 4, [70, 100, None, None]), (0, 0, None), (89, 4, [None, 40, 10, None]), None, None])
Step 2 ([(91, 4, [None, None, 60, 30]), (0, 0, None), (0, 0, None), (89, 4, [None, None, 50, 20]), (76, 4, [70, 100, None, None]), (0, 0, None), (89, 4, [None, 40, 10, None]), (48, 1, [90]), None])
------------
6
from query self.level2_tables -> [(91, 4, [None, None, 60, 30]), (

In [86]:
sum_squares = sum(len(b)**2 for b in selfbuckets)

In [31]:
selfbuckets = [[1,2,3,4,5], [1], [1], [3]]

In [33]:
sum_squares

28